In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
import os
import json
import requests
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import fitz  # PyMuPDF

PDF_FOLDER = '/home/ttran/code/ai_pdf_reader/data'
OLLAMA_URL = 'http://ollamacuda.home:11434/api/chat'
MODEL_NAME = 'qwen3.5:9b'  # Or a text model like 'qwen2.5:7b'
MAX_WORKERS = 4

KEYS_TO_EXTRACT = [
    'Date of Sale',
    'Earliest Time Sale Will Begin',
    'Place of Sale of Property',
    'Legal Description',
    'Date Added',
    'Physical Address'
]

PROMPT_TEMPLATE = """You are extracting data from a property sale notice document.
Extract the following fields from the document text below:
{keys}

Rules:
- "Physical Address" is the property's physical/street address (sometimes labeled "commonly known as").
- Return ONLY a valid JSON object with exactly these keys: 
{key_list}.
- If a field is not present in the document, set its value to null.
- Do not include any text outside the JSON object.

Document Text:
{pdf_text}
"""

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract all text from a PDF file using PyMuPDF."""
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text() + "\n"
    return text

def query_ollama(pdf_text: str, file_name: str) -> dict:
    """Send extracted text to Ollama's chat API and get back structured JSON."""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": PROMPT_TEMPLATE.format(
                    keys="\n".join(f"- {k}" for k in KEYS_TO_EXTRACT),
                    key_list=json.dumps(KEYS_TO_EXTRACT),
                    pdf_text=pdf_text
                ),
            }
        ],
        "stream": False,
        "format": "json",
        "think": True,
        "options": {"temperature": 0, "num_predict": 32000},
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=300)
        response.raise_for_status()
        content = response.json()['message']['content']
        return json.loads(content)
    except Exception as e:
        print(f"Error querying Ollama for {file_name}: {e}")
        return {}

def process_pdf(file_path: str) -> dict:
    file_name = Path(file_path).stem
    pdf_text = extract_text_from_pdf(file_path)
    extracted_data = query_ollama(pdf_text, file_name)
    return {file_name: {k: extracted_data.get(k) for k in KEYS_TO_EXTRACT}}

def main() -> list:
    pdf_files = sorted(
        os.path.join(PDF_FOLDER, f)
        for f in os.listdir(PDF_FOLDER)
        if f.lower().endswith('.pdf')
    )
    print(f"Processing {len(pdf_files)} PDFs...")

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_pdf, pdf): pdf for pdf in pdf_files}
        for future in futures:
            try:
                results.append(future.result())
            except Exception as e:
                print(f"Task failed for {futures[future]}: {e}")

    return results

if __name__ == "__main__":
    final_results = main()
    print(json.dumps(final_results, indent=2))

/home/ttran/miniconda3/envs/mypython312/lib/python3.12/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Processing 10 PDFs...
